In [1]:
from src.datasets.loaders import MetaICLDataset, make_meta_icl_collator
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

/home/mortega/repositories/graph-representation-learning-for-llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = MetaICLDataset("data/graphqa/meta-icl/standard/train.json", k=5)

In [17]:
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
loader = DataLoader(dataset, batch_size=50, collate_fn=make_meta_icl_collator(tokenizer))

In [18]:
batch = next(iter(loader))
batch["input_ids"].shape, batch["labels"].shape

(torch.Size([50, 1024]), torch.Size([50, 1024]))

In [19]:
# Verify masking: only the answer tokens should be unmasked (labels != -100).
for i in range(batch["input_ids"].shape[0]):
    ids = batch["input_ids"][i]
    labels = batch["labels"][i]
    unmasked = ids[labels != -100]
    decoded = tokenizer.decode(unmasked)
    expected = dataset[i]["answer"]
    print(f"[{i}] unmasked tokens: {unmasked.tolist()}")
    print(f"[{i}] decoded answer: {decoded!r}  (expected suffix: {expected!r})")
    assert (labels[batch["attention_mask"][i] == 0] == -100).all(), "padding not masked"
    assert decoded.strip() == expected.strip(), "unmasked tokens != answer"
print("All checks passed.")

[0] unmasked tokens: [3363, 13]
[0] decoded answer: ' Yes.'  (expected suffix: 'Yes.')
[1] unmasked tokens: [362, 13]
[1] decoded answer: ' 2.'  (expected suffix: '2.')
[2] unmasked tokens: [1478, 13]
[2] decoded answer: ' 14.'  (expected suffix: '14.')
[3] unmasked tokens: [352, 13]
[3] decoded answer: ' 1.'  (expected suffix: '1.')
[4] unmasked tokens: [2242, 13]
[4] decoded answer: ' 23.'  (expected suffix: '23.')
[5] unmasked tokens: [362, 13]
[5] decoded answer: ' 2.'  (expected suffix: '2.')
[6] unmasked tokens: [642, 13]
[6] decoded answer: ' 5.'  (expected suffix: '5.')
[7] unmasked tokens: [3363, 13]
[7] decoded answer: ' Yes.'  (expected suffix: 'Yes.')
[8] unmasked tokens: [3363, 13]
[8] decoded answer: ' Yes.'  (expected suffix: 'Yes.')
[9] unmasked tokens: [3363, 13]
[9] decoded answer: ' Yes.'  (expected suffix: 'Yes.')
[10] unmasked tokens: [24652, 13]
[10] decoded answer: ' 153.'  (expected suffix: ' 153.')
[11] unmasked tokens: [513, 13]
[11] decoded answer: ' 3.'  (ex

In [14]:
batch

{'input_ids': tensor([[50256, 50256, 50256,  ...,    25,  3363,    13],
         [    8,   357,    20,  ...,    25,   362,    13]]),
 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'labels': tensor([[-100, -100, -100,  ..., -100, 3363,   13],
         [-100, -100, -100,  ..., -100,  362,   13]])}